# 71 — Simulation-based calibration (paper Section 7.1)

**Question.** Does the sampler draw from the posterior it claims to? One simulated data set cannot answer this: a single estimate can land near the truth by chance, or far from it with nothing wrong. Simulation-based calibration (SBC; Talts, Betancourt, Simpson, Vehtari & Gelman, 2018) answers it with a property that holds only for a correct sampler.

**Premise.** If θ̃ is drawn from the prior and data y from the model given θ̃, then θ̃ is a draw from the posterior p(θ | y). So among L independent posterior draws, the rank of θ̃ (how many draws fall below it) is uniform on 0, …, L. Repeating the cycle many times, a correct sampler gives flat rank histograms for every parameter; an error shows as a U shape (posterior too narrow), a hump (too wide) or a slope (biased).

**Models.** Three small trees chosen to exercise different code paths (`sbc_models.py`):

| Model | Tree | Nodes | Code paths |
|---|---|---|---|
| S1 | binary disengagement | probit Rasch with a person covariate; probit 2PL | exact draws of θ, b, a, β; LKJ correlation (MH); σ² estimated on node 1's dimension, fixed at 1 on node 2's |
| S2 | polytomous disengagement | probit Rasch; graded response with an item covariate | GRM thresholds (exact draw plus Cowles step), item covariate with interweaving |
| S3 | binary disengagement | logit 2PL; probit 3PL | logit MH updates, 3PL indicators and exact c draws |

**Priors.** SBC is valid for any proper prior, but the simulating prior and the fitting prior must be the same. The SBC priors are proper and moderately informative (variances InvGamma(4, 3), mean 1), so the simulated tests look like real ones; the slope, asymptote and correlation priors are the package defaults.

**Runtime.** Step 3 times one replication per model and projects the full run before you start it. Results are written after every replication, so an interrupted run resumes where it stopped.

## Setup

In [ ]:
# Install the package from this repository (editable), once per environment.
# The notebook lives in replication/<folder>/, so the repository root is two levels up.
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1]
!{sys.executable} -m pip install -q -e "{ROOT}[full]"

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from concurrent.futures import ProcessPoolExecutor, as_completed
import irtree
import sbc_models as sbc                 # models, prior draws, simulator, one replication
pd.set_option("display.width", 140)

RESULTS = pathlib.Path("results"); RESULTS.mkdir(exist_ok=True)

# Runtime log: wrap every expensive step in `with timed("label"):`.
# The last cell prints the table and saves it to results/.
import time, contextlib
RUNTIME = []

@contextlib.contextmanager
def timed(step):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    RUNTIME.append({"step": step, "seconds": round(dt, 1)})
    print(f"[{step}] " + (f"{dt / 60:.1f} min" if dt >= 60 else f"{dt:.1f} s"))

## Settings
* `P`, `I`: persons and items per simulated data set (small on purpose: SBC needs many fits).
* `N_ITER`: iterations per fit, one chain; warm-up is chosen by the package (evidence mode). 8000 keeps the bulk ESS of the slowest parameters (2PL slopes, the correlation) above `L` in most fits; at 2000 the slopes of S1 had ESS of about 30.
* `L`: thinned posterior draws per fit; ranks run from 0 to `L`. The draws should be close to independent, so the notebook reports the effective sample size and flags fits with ESS < `L` (Step 6).
* `N_REPS`: replications per model. 300 gives rank histograms with 20 bins of about 15 counts.
* `N_WORKERS`: replications run in parallel processes.

In [ ]:
P, I     = 200, 10
N_ITER   = 8000
L        = 99
N_REPS   = 300
MODELS   = ["S1", "S2", "S3"]
N_WORKERS = max(1, (os.cpu_count() or 2) - 1)
SEED     = 2026
OUT      = RESULTS / "sbc_ranks.csv"
print(f"P={P} I={I} N_ITER={N_ITER} L={L} N_REPS={N_REPS} workers={N_WORKERS}")

## Step 1 — One replication by hand (model S1)
The four SBC steps made visible: draw from the prior, simulate, fit, rank.

In [ ]:
rng = np.random.default_rng(SEED)
prm, truth = sbc.draw_prior("S1", P, I, rng)
print("true values of the tracked parameters:")
pd.Series(truth).round(3)

In [ ]:
Y = sbc.simulate("S1", prm, rng)
print("category counts (0 = rapid guess, 1 = incorrect, 2 = correct):", np.bincount(Y.ravel()))
spec, nodes = sbc.spec_for("S1"), sbc.nodes_for("S1")
data = irtree.expand_data(Y, spec, categories_start=0, X=prm["X"], X_names=["x1"])
priors = sbc.make_priors("S1", spec, nodes)
priors.summary()

In [ ]:
with timed("one S1 fit"):
    res = irtree.run_chain(data, nodes, priors, n_iter=N_ITER, spec=spec, seed=1, progress=False)
chain = res.chain.astype(float)
keep = np.linspace(0, chain.shape[0] - 1, L).round().astype(int)
pd.DataFrame([{"param": k, "true": v,
               "post_mean": chain[:, res.param_names.index(k)].mean(),
               "rank (0..L)": int((chain[keep, res.param_names.index(k)] < v).sum())}
              for k, v in truth.items()]).round(3)

## Step 2 — The same replication through the worker function
`sbc.one_replication` does Steps 1's four parts in one call and returns one row per tracked parameter. This is what the full run calls.

In [ ]:
pd.DataFrame(sbc.one_replication("S1", 0, P, I, N_ITER, L, SEED)).round(3)

## Step 3 — Pilot timing and projected runtime
One replication per model, then the projection: seconds per replication × `N_REPS` / `N_WORKERS`, summed over models. Parallel workers share the machine, so the real time can be somewhat longer.

In [ ]:
pilot = {}
for m in MODELS:
    with timed(f"pilot {m}"):
        rows = sbc.one_replication(m, 10_000, P, I, N_ITER, L, SEED)
    pilot[m] = rows[0]["seconds"]
proj = sum(pilot.values()) * N_REPS / N_WORKERS
print({m: f"{s:.1f} s" for m, s in pilot.items()})
print(f"projected full run: {proj / 60:.0f} min ({proj / 3600:.1f} h) on {N_WORKERS} workers")

## Step 4 — Full run (resumable)
Each finished replication is appended to `results/sbc_ranks.csv`. Re-running the cell skips the replications already there. Lower `N_REPS` above for a shorter first pass; increase it later and the run continues.

In [ ]:
done = set()
if OUT.exists():
    prev = pd.read_csv(OUT)
    done = set(zip(prev["model"], prev["rep"]))
todo = [(m, r) for m in MODELS for r in range(N_REPS) if (m, r) not in done]
print(f"{len(done)} replications on file, {len(todo)} to run")

with timed(f"SBC run ({len(todo)} replications)"):
    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {pool.submit(sbc.one_replication, m, r, P, I, N_ITER, L, SEED): (m, r) for m, r in todo}
        for k, fut in enumerate(as_completed(futures), 1):
            pd.DataFrame(fut.result()).to_csv(OUT, mode="a", header=not OUT.exists(), index=False)
            if k % 25 == 0 or k == len(todo):
                print(f"  {k}/{len(todo)} done")

## Step 5 — Load the ranks

In [ ]:
ranks = pd.read_csv(OUT)
print(ranks.groupby("model")["rep"].nunique().rename("replications"))
print(ranks.groupby("model")["seconds"].describe()[["mean", "50%", "max"]].round(1))

## Step 6 — Are the thinned draws close to independent?
Ranks computed from autocorrelated draws are not uniform even for a correct sampler. The table gives, per parameter, the median bulk ESS of the retained chain and the share of fits with ESS < `L`. If that share is large for a parameter, increase `N_ITER` for that model; Step 8 also repeats the test on the fits with ESS ≥ `L` only.

In [ ]:
ess_tab = (ranks.assign(low=ranks["ess"] < L)
                .groupby(["model", "param"])
                .agg(median_ess=("ess", "median"), share_ess_below_L=("low", "mean")))
ess_tab.round(2)

## Step 7 — Rank histograms
20 bins of 5 ranks each. The grey band is the 99% interval for a bin count under uniformity (Binomial(N, 1/20)); with 20 bins per panel, about one bin in five panels falls outside it by chance.

In [ ]:
def rank_panels(df, model, bins=20, ax_row=None):
    sub = df[df["model"] == model]
    params = list(dict.fromkeys(sub["param"]))
    ncol = 4
    nrow = int(np.ceil(len(params) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(3.2 * ncol, 2.4 * nrow), squeeze=False)
    for ax, p in zip(axes.flat, params):
        r = sub.loc[sub["param"] == p, "rank"].to_numpy()
        n = len(r)
        lo, hi = stats.binom.ppf([0.005, 0.995], n, 1 / bins)
        ax.axhspan(lo, hi, color="0.88")
        ax.axhline(n / bins, color="0.6", lw=0.8)
        ax.hist(r, bins=np.linspace(0, L + 1, bins + 1), color="#3b6ea5", edgecolor="white")
        ax.set_title(p, fontsize=9); ax.set_yticks([]); ax.set_xticks([0, L + 1])
    for ax in list(axes.flat)[len(params):]:
        ax.axis("off")
    fig.suptitle(f"{model}: {sbc.MODELS[model]}", fontsize=10)
    fig.tight_layout()
    return fig

for m in MODELS:
    fig = rank_panels(ranks, m)
    fig.savefig(RESULTS / f"sbc_ranks_{m}.png", dpi=200)
    plt.show()

## Step 8 — Uniformity tests
A chi-square test of the 20 bin counts per parameter, on all fits and on the fits with ESS ≥ `L`. With about 30 tests, a few p-values below 0.05 are expected by chance; a parameter whose p-value is tiny in both columns and whose histogram has a clear shape points to a problem.

In [ ]:
def chi2_p(r, bins=20):
    counts, _ = np.histogram(r, bins=np.linspace(0, L + 1, bins + 1))
    return stats.chisquare(counts).pvalue

unif = (ranks.groupby(["model", "param"])
             .apply(lambda g: pd.Series({
                 "n": len(g),
                 "p_all": chi2_p(g["rank"]),
                 "n_ess_ok": int((g["ess"] >= L).sum()),
                 "p_ess_ok": chi2_p(g.loc[g["ess"] >= L, "rank"]) if (g["ess"] >= L).sum() >= 40 else np.nan})))
unif.to_csv(RESULTS / "sbc_uniformity.csv")
unif.round(3)

## Step 9 — Figure for the paper
A selection of panels across the three models: one location, one slope or asymptote, one variance, the correlation, one covariate effect.

In [ ]:
SELECT = [("S1", "b_2_1"), ("S1", "a_2_1"), ("S1", "beta_1_x1"), ("S1", "rho_1_2"),
          ("S2", "b_2_1_thresh_1"), ("S2", "kappa_2_q1"), ("S2", "sigma2_2"), ("S2", "b.variance_2"),
          ("S3", "a_1_1"), ("S3", "c_2_1"), ("S3", "b_2_1"), ("S3", "rho_1_2")]
fig, axes = plt.subplots(3, 4, figsize=(12, 7))
for ax, (m, p) in zip(axes.flat, SELECT):
    r = ranks.loc[(ranks["model"] == m) & (ranks["param"] == p), "rank"].to_numpy()
    lo, hi = stats.binom.ppf([0.005, 0.995], len(r), 1 / 20)
    ax.axhspan(lo, hi, color="0.88"); ax.axhline(len(r) / 20, color="0.6", lw=0.8)
    ax.hist(r, bins=np.linspace(0, L + 1, 21), color="#3b6ea5", edgecolor="white")
    ax.set_title(f"{m}: {p}", fontsize=9); ax.set_yticks([]); ax.set_xticks([0, L + 1])
fig.tight_layout()
fig.savefig(RESULTS / "sbc_figure_paper.png", dpi=300)

## Runtime

In [ ]:
rt = pd.DataFrame(RUNTIME)
rt.loc[len(rt)] = {"step": "TOTAL", "seconds": rt["seconds"].sum()}
rt.to_csv(RESULTS / "runtime_71.csv", index=False)
rt